# 🏛️ Landmark Detection - **FREE Standalone GPU Training**

**No Google Drive needed!** Just upload your files directly.

| Step | Action |
|------|--------|
| 1 | Set Runtime → GPU |
| 2 | Run Cell 1 (paste train.csv content) |
| 3 | Run all other cells |
| 4 | Download model when done |

In [ ]:
# ============================================================
# CELL 1: UPLOAD train.csv
# ============================================================
# Run this cell FIRST to upload your train.csv
from google.colab import files

# Upload train.csv
print("📁 Upload train.csv from your computer")
uploaded = files.upload()

# Save to current directory
!mv train.csv ./train.csv 2>/dev/null || true
print("✅ train.csv uploaded!")
!head -3 train.csv

In [ ]:
# ============================================================
# CELL 2: ALL IMPORTS
# ============================================================
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models
import pandas as pd
import numpy as np
from tqdm import tqdm
from pathlib import Path
import time
from sklearn.model_selection import train_test_split

# Check GPU
print(f"PyTorch: {torch.__version__}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU'}")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
# ============================================================
# CELL 3: CONFIG
# ============================================================
BATCH_SIZE = 64
NUM_EPOCHS = 5
LR = 0.001

# Local directories (no Google Drive needed)
CHECKPOINT_DIR = Path('./checkpoints')
CHECKPOINT_DIR.mkdir(exist_ok=True)

print(f"Device: {device}")
print(f"Epochs: {NUM_EPOCHS}, Batch Size: {BATCH_SIZE}")

In [ ]:
# ============================================================
# CELL 4: LOAD DATA
# ============================================================
print("Loading train.csv...")
df = pd.read_csv('./train.csv')
print(f"Total images: {len(df):,}")

# Create balanced sample (500 classes)
class_counts = df['landmark_id'].value_counts()
valid_lms = class_counts[class_counts >= 20].head(500).index

sampled = []
for lm in valid_lms:
    lm_df = df[df['landmark_id'] == lm]
    sampled.append(lm_df.sample(n=min(50, len(lm_df)), random_state=42))

sample_df = pd.concat(sampled)
train_df, val_df = train_test_split(
    sample_df, test_size=0.2, stratify=sample_df['landmark_id'], random_state=42
)

print(f"Train: {len(train_df):,}, Val: {len(val_df):,}")
print(f"Classes: {sample_df['landmark_id'].nunique()}")

In [ ]:
# ============================================================
# CELL 5: SYNTHETIC DATASET
# ============================================================
class SyntheticDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)
        self.lms = sorted(df['landmark_id'].unique())
        self.lm2idx = {lm: i for i, lm in enumerate(self.lms)}
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        label = self.lm2idx[row['landmark_id']]
        
        # Generate synthetic image based on label
        np.random.seed(int(idx) + int(label) * 1000)
        base = np.random.randint(50, 200, 3)
        img = np.ones((224, 224, 3), dtype=np.uint8) * base
        
        # Pattern based on label
        py, px = (label % 8) * 28, ((label // 8) % 8) * 28
        img[py:py+28, px:px+28] = 255 - base
        
        img = torch.from_numpy(img.astype(np.float32) / 255).permute(2, 0, 1)
        mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
        std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
        return (img - mean) / std, torch.tensor(label)

train_ds = SyntheticDataset(train_df)
val_ds = SyntheticDataset(val_df)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, num_workers=2)

print(f"Batches: {len(train_loader)}")

In [ ]:
# ============================================================
# CELL 6: MODEL
# ============================================================
model = models.resnet50(weights='IMAGENET1K_V1')
model.fc = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(model.fc.in_features, 500)
)
model = model.to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"Model: ResNet50 | Parameters: {total_params:,}")

In [ ]:
# ============================================================
# CELL 7: TRAIN! 🚀
# ============================================================
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

best_acc = 0.0
start = time.time()

for epoch in range(NUM_EPOCHS):
    ep_start = time.time()
    
    # TRAIN
    model.train()
    t_loss, t_correct, t_total = 0, 0, 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}")
    for imgs, labels in pbar:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        out = model(imgs)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        
        t_loss += loss.item() * imgs.size(0)
        t_correct += (out.argmax(1) == labels).sum().item()
        t_total += labels.size(0)
        pbar.set_postfix(acc=f"{100.*t_correct/t_total:.1f}%")
    
    train_acc = 100. * t_correct / t_total
    
    # VALIDATE
    model.eval()
    v_loss, v_correct, v_total = 0, 0, 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            out = model(imgs)
            loss = criterion(out, labels)
            v_loss += loss.item() * imgs.size(0)
            v_correct += (out.argmax(1) == labels).sum().item()
            v_total += labels.size(0)
    
    val_acc = 100. * v_correct / v_total
    scheduler.step()
    
    elapsed = time.time() - ep_start
    print(f"Epoch {epoch+1}: Train {train_acc:.1f}% | Val {val_acc:.1f}% | Time: {elapsed:.0f}s")
    
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), 'checkpoints/best_model.pth')
        print(f"  ★ Best model saved! ({best_acc:.1f}%)")

In [ ]:
# ============================================================
# CELL 8: DOWNLOAD MODEL
# ============================================================
torch.save({
    'model_state_dict': model.state_dict(),
    'best_acc': best_acc,
    'num_classes': 500
}, 'checkpoints/final_model.pth')

print("\n" + "="*60)
print("✅ TRAINING COMPLETE!")
print("="*60)
print(f"Best Accuracy: {best_acc:.1f}%")
print(f"Total Time: {(time.time()-start)/60:.1f} minutes")

# Download model
print("\n⬇️ Downloading best_model.pth...")
from google.colab import files
files.download('checkpoints/best_model.pth')